# 🚀 Road2AI: Hybrid RAG + BGE-Reranker (Peak Optimization)

Notebook này được thiết kế ĐẶC BIỆT để tối ưu hóa quy trình trích xuất tài liệu pháp lý, giúp hệ thống đạt điểm F2 Score cao nhất (Target: 0.45 - 0.50+).

### 🌟 Các Cải Tiến Mới Nhất:
- **Bộ Reranker Mạnh Mẽ (BGE-M3):** Đánh giá lại độ chính xác của các điều luật bằng Cross-Encoder.
- **Bộ Chống Lỗi Trích Dẫn Âm Thầm (Anti-Silent Discard):** Tự động đồng bộ và viết lại phần `Cơ sở pháp lý tham chiếu` trong câu trả lời để Kaggle không trừ điểm oan.
- **Chống Cache Google Drive:** Sinh ra file Zip có gắn Timestamp để đảm bảo tải đúng file kết quả mới nhất.
- **Tự động Resume:** Khôi phục thông minh từ Drive nếu Colab bị mất kết nối giữa chừng.

# 🚀 R2AI2026 — Phương án A+B: Hybrid Search + BGE-Reranker
**Mục tiêu:** Tăng điểm từ 0.3976 lên 0.45-0.50 | **Không cần chạy lại LLM!**

### Tính năng chống mất dữ liệu:
- **Cell 6 (Retrieve):** Background thread tự động backup `retrieval.db` lên Drive **mỗi 5 phút**
- **Cell 7 (Rerank):** Auto-checkpoint mỗi 50 câu → Drive. Nếu bị ngắt, **chạy lại sẽ resume từ câu dở**

### 📁 File cần có trong Drive > `R2AI_Artifacts`:
- `sme_rag_clean.zip` (71KB) ✅
- `results_partial.jsonl` (6MB) ✅
- `R2AIStage1DATA.json` (518KB) ✅

## ⚙️ Cell 1 — Kết nối Google Drive

*Hành động: Bấm chạy Cell này để gắn (mount) Google Drive vào hệ thống. Việc này rất quan trọng để lưu trữ kết quả và checkpoint.*

In [ ]:
import subprocess, os
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('🖥️  GPU:', r.stdout.strip() or '❌ KHÔNG CÓ GPU!')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts_Test'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'\n📋 Scan {DRIVE_DIR}:')
for root, dirs, files in os.walk(DRIVE_DIR):
    level = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * level
    rel = root.replace(DRIVE_DIR, '') or '/'
    print(f'{indent}📁 {rel}/')
    for f in files:
        sz = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f'{indent}  📄 {f} ({sz:.1f}MB)')
print('\n✅ Cell 1 Done!')

## 📥 Cell 2 — Tải Source Code từ GitHub

*Hành động: Clone mã nguồn mới nhất từ nhánh `experiment-rag-upgrade`. Chứa tất cả các bản vá chống lỗi trích dẫn và tối ưu hóa thư viện.*

In [ ]:
import subprocess, sys
print('📦 Cài thư viện...')
pkgs = [['datasets', 'huggingface_hub'], ['chromadb==0.5.23'],
        ['sentence-transformers>=2.7.0'], ['rank-bm25', 'numpy', 'tqdm']]
for pkg_list in pkgs:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkg_list
    print(f'  {" ".join(pkg_list)}...', end=' ')
    r = subprocess.run(cmd, capture_output=True, text=True)
    print('✅' if r.returncode == 0 else f'⚠️ {r.stderr[-80:]}')
print('\n✅ Cell 2 Done!')

## 📦 Cell 3 — Cài đặt Thư Viện (Pip Install)

*Hành động: Cài đặt các thư viện lõi cho AI (PyTorch, Transformers, Sentence-Transformers, ChromaDB). Có thể mất 1-2 phút.*

In [ ]:
import os, sys, shutil, subprocess

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts_Test'
WORK_DIR  = '/content/sme-legal-assistant'
REPO_URL  = 'https://github.com/Platypus27-coder/sme-legal-assistant.git'
BRANCH    = 'experiment-rag-upgrade'

if os.path.exists(WORK_DIR): 
    shutil.rmtree(WORK_DIR)

print(f'⬇️ Clone branch {BRANCH} từ Github...')
r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK_DIR], capture_output=True, text=True)
if r.returncode != 0:
    print(f'❌ Lỗi Clone: {r.stderr}')
    raise SystemExit

for d in ['output', 'cache', 'raw', 'index']:
    os.makedirs(f'{WORK_DIR}/artifacts/{d}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

# Khôi phục tệp câu hỏi gốc
if os.path.exists(f'{DRIVE_DIR}/R2AIStage1DATA.json'):
    shutil.copy2(f'{DRIVE_DIR}/R2AIStage1DATA.json', f'{WORK_DIR}/data/R2AIStage1DATA.json')
    print('✅ Khôi phục thành công file câu hỏi R2AIStage1DATA.json')

# Khôi phục kết quả sinh một phần cũ (nếu có) để chạy tiếp tục (Checkpoint)
if os.path.exists(f'{DRIVE_DIR}/results_partial_test.jsonl'):
    shutil.copy2(f'{DRIVE_DIR}/results_partial_test.jsonl', f'{WORK_DIR}/artifacts/output/results_partial_test.jsonl')
    print('✅ [Checkpoint] Khôi phục kết quả đang chạy dở: results_partial_test.jsonl')

sys.path.insert(0, f'{WORK_DIR}/src')
os.chdir(WORK_DIR)

from vpl.settings import SEARCH
print(f'\n✅ Cấu hình hệ thống hiện tại: high_conf={SEARCH.high_conf_threshold}, max_art={SEARCH.max_articles}')
print('✅ Cell 3 Done!')


## 🧠 Cell 4 — Tải Embedding Model & LLM

*Hành động: Tải trước mô hình `keepitreal/vietnamese-sbert` (Embedding) và `BGE-Reranker-Large` vào bộ nhớ.*

In [ ]:
import os, shutil, subprocess
DRIVE_DIR    = '/content/drive/MyDrive/R2AI_Artifacts'
CHUNKS_LOCAL = 'artifacts/raw/chunks.jsonl'

# Tìm chunks từ Drive (từ lần chạy trước)
candidates = [f'{DRIVE_DIR}/raw/chunks.jsonl', f'{DRIVE_DIR}/chunks.jsonl']
found = next((c for c in candidates if os.path.exists(c)), None)

if found:
    shutil.copy2(found, CHUNKS_LOCAL)
    n = sum(1 for _ in open(CHUNKS_LOCAL, encoding='utf-8'))
    print(f'✅ Chunks từ Drive: {n:,} chunks — Bỏ qua Ingest!')
else:
    print('🔄 Tải dữ liệu luật từ HuggingFace (~10-15 phút)...')
    result = subprocess.run(['python', 'run.py', 'ingest'], capture_output=False)
    if result.returncode == 0 and os.path.exists(CHUNKS_LOCAL):
        n = sum(1 for _ in open(CHUNKS_LOCAL, encoding='utf-8'))
        print(f'✅ {n:,} chunks')
        shutil.copy2(CHUNKS_LOCAL, f'{DRIVE_DIR}/chunks.jsonl')
        print('☁️  Backed up to Drive')
    else:
        print('❌ Ingest thất bại!')
print('\n✅ Cell 4 Done!')

## 🗃️ Cell 5 — Build Database (BM25 + ChromaDB)

*Hành động: Xây dựng cơ sở dữ liệu tìm kiếm. Nếu bạn đã có file DB trên Drive, hệ thống sẽ tự động phục hồi mà không cần xây lại.*

In [ ]:
import os, shutil, subprocess
DRIVE_DIR   = '/content/drive/MyDrive/R2AI_Artifacts_Test'
BM25_LOCAL  = 'artifacts/index/bm25/corpus.pkl'
CHROMA_DB   = 'artifacts/index/chroma/chroma.sqlite3'
DRIVE_TAR   = f'{DRIVE_DIR}/index_built_test.tar.gz'

# Khôi phục Index RAG mới đã build từ Drive (nếu có) để tiết kiệm thời gian khi chạy lại
if os.path.exists(DRIVE_TAR) and not os.path.exists(BM25_LOCAL):
    print('📦 [Checkpoint] Khôi phục Index làm giàu văn cảnh mới từ Drive...')
    os.makedirs('artifacts', exist_ok=True)
    subprocess.run(['tar', '-xzf', DRIVE_TAR, '-C', 'artifacts'], capture_output=True)

if not os.path.exists(BM25_LOCAL) or not os.path.exists(CHROMA_DB):
    print('🔄 Bắt đầu xây dựng mới Index (BM25 + ChromaDB) với Làm giàu văn cảnh — ~45 phút...')
    if os.path.exists('artifacts/index'):
        shutil.rmtree('artifacts/index')
    # Thêm check=True để quẳng lỗi đỏ ra màn hình Colab nếu tiến trình index bị sập, tránh silent fail
    subprocess.run(['python', 'run.py', 'index', '--device', 'cuda', '--reset'], capture_output=False, check=True)
    
    # Sao lưu lên Drive ngay sau khi build xong để dùng cho các lần sau
    if os.path.exists(BM25_LOCAL):
        print('\n☁️ Sao lưu index mới lên Drive để tái sử dụng...')
        os.makedirs(DRIVE_DIR, exist_ok=True)
        subprocess.run(['tar', '-czf', DRIVE_TAR, '-C', 'artifacts', 'index'], capture_output=True)
        sz = os.path.getsize(DRIVE_TAR) / 1024 / 1024
        print(f'☁️ index_built_test.tar.gz ({sz:.0f}MB) đã được lưu an toàn!')

for p, label in [(BM25_LOCAL,'BM25'), (CHROMA_DB,'ChromaDB')]:
    ok = os.path.exists(p)
    sz = f'{os.path.getsize(p)/1024/1024:.0f}MB' if ok else ''
    print(f'  {"✅" if ok else "❌"} {label} {sz}')
print('\n✅ Cell 5 Done!')


## 🔍 Cell 6 — Hybrid Retrieve (Tìm kiếm lai)

*Hành động: Tìm kiếm trước top 20 tài liệu liên quan cho mỗi câu hỏi. Chạy liên tục cho 2000 câu và tự động lưu checkpoint mỗi 5 phút.*

In [ ]:
# Cell 6: Hybrid Retrieve với background backup thread
import os, shutil, sqlite3, subprocess, threading, time

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts_Test'
DB_LOCAL  = 'artifacts/cache/retrieval_test.db'
DRIVE_DB  = f'{DRIVE_DIR}/retrieval_hybrid_test.db'
BACKUP_INTERVAL = 300  # 5 phút sao lưu một lần

# Khôi phục DB từ Drive nếu đã có sẵn tiến trình cũ (Checkpoint)
if os.path.exists(DRIVE_DB) and not os.path.exists(DB_LOCAL):
    shutil.copy2(DRIVE_DB, DB_LOCAL)
    try:
        n = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
        print(f'📦 [Checkpoint] Đã khôi phục tiến trình retrieval_test.db từ Drive ({n}/2000 câu)')
    except:
        os.remove(DB_LOCAL)
        print('⚠️ DB khôi phục bị lỗi, bắt đầu lại')

done = 0
if os.path.exists(DB_LOCAL):
    try:
        done = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
    except: done = 0

if done >= 2000:
    print(f'✅ Đã retrieve đủ 2000/2000 câu — Bỏ qua!')
else:
    print(f'🔄 Bắt đầu chạy Hybrid Retrieve: còn {2000-done} câu cần tìm kiếm...')
    print(f'☁️ Tự động sao lưu tiến trình lên Drive mỗi {BACKUP_INTERVAL//60} phút')

    # Background backup thread
    stop_backup = threading.Event()
    def _backup_loop():
        count = 0
        while not stop_backup.is_set():
            stop_backup.wait(BACKUP_INTERVAL)
            if stop_backup.is_set(): break
            if os.path.exists(DB_LOCAL):
                try:
                    shutil.copy2(DB_LOCAL, DRIVE_DB)
                    n = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
                    count += 1
                    print(f'   ☁️ [Auto-backup #{count}] {n}/2000 câu đã đồng bộ -> Drive')
                except Exception as e:
                    print(f'   ⚠️ Đồng bộ thất bại: {{e}}')

    backup_thread = threading.Thread(target=_backup_loop, daemon=True)
    backup_thread.start()

    try:
        # Thêm check=True để quẳng lỗi đỏ ra màn hình Colab nếu tiến trình retrieve bị sập
        subprocess.run(
            ['python', 'run.py', 'retrieve',
             '--questions', 'data/R2AIStage1DATA.json',
             '--device', 'cuda'],
            capture_output=False,
            check=True
        )
    finally:
        stop_backup.set()

    if os.path.exists(DB_LOCAL):
        shutil.copy2(DB_LOCAL, DRIVE_DB)
        n = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
        print(f'\n☁️ Sao lưu lần cuối thành công: {n}/2000 câu đã lưu an toàn!')

print('\n✅ Cell 6 Done!')


## 🏆 Cell 7 — BGE-Reranker + Viết lại câu trả lời

*Hành động: Đây là bước QUAN TRỌNG NHẤT. Sử dụng mô hình BGE để chấm điểm độ chuẩn xác, loại bỏ các điều luật nhiễu (noise) bằng cấu hình đỉnh cao (HIGH=0.68, MAX=2).* 
Đồng thời tự động đồng bộ trích dẫn để lấy trọn điểm F2 từ Kaggle.

*(Hệ thống sẽ tự động sinh ra file Zip Độc Nhất (kèm thời gian) để tải về không bị lỗi).* 
**Lưu ý:** Nếu muốn chạy lại từ đầu, hãy thêm chữ `--reset` vào cuối câu lệnh subprocess.

In [ ]:
# Cell 7: BGE-Reranker + Rewrite Answers
import os, shutil, subprocess, glob

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts_Test'
ckpt = f'{DRIVE_DIR}/results_reranked_checkpoint_test.json'

# Khôi phục checkpoint nếu có sẵn tiến trình cũ (Checkpoint)
if os.path.exists(ckpt):
    import json
    try:
        done_count = len(json.loads(open(ckpt, encoding='utf-8').read()))
        print(f'🔄 [Checkpoint] Tìm thấy checkpoint đang chạy dở: {done_count}/2000 câu -> TIẾP TỤC CHẠY!')
    except:
        print('⚠️ Checkpoint cũ bị lỗi, bắt đầu sinh lại từ đầu')
else:
    print('🆕 Chạy mới hoàn toàn (chưa có checkpoint trên Drive)')

print('🎯 THAM SỐ ĐỈNH CAO: HIGH_CONF=0.68 | SAFE=0.58 | MAX_ART=2')
print('☁️ Tự động lưu checkpoint mỗi 50 câu lên Drive')

# Thêm check=True để quẳng lỗi đỏ ra màn hình Colab nếu tiến trình sinh bị sập
subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', '0.68', '--safe', '0.58',
     '--min-art', '0', '--max-art', '2',
     '--device', 'cuda', '--batch-size', '64',
     '--checkpoint-every', '50'],
    capture_output=False,
    check=True
)

# Tìm file zip mới nhất do rerank_retune.py sinh ra
zip_files = glob.glob('artifacts/output/submission_reranked_test_*.zip')
if not zip_files:
    zip_files = glob.glob('artifacts/output/submission_reranked_test.zip')

if zip_files:
    # Lấy file zip mới nhất (sắp xếp theo thời gian)
    out_zip = sorted(zip_files, key=os.path.getmtime)[-1]
    sz = os.path.getsize(out_zip) / 1024 / 1024
    zip_name = os.path.basename(out_zip)
    drive_zip_path = f'{DRIVE_DIR}/{zip_name}'
    shutil.copy2(out_zip, drive_zip_path)
    print(f'\n🏆 {zip_name} ({sz:.1f}MB) đã được xuất thành công!')
    print(f'☁️ Đường dẫn trên Drive: {drive_zip_path}')
    print('\n📥 HÃY VÀO GOOGLE DRIVE, TẢI FILE ZIP NÀY VỀ VÀ NỘP LÊN KAGGLE ĐỂ BỨT PHÁ ĐIỂM SỐ!')
else:
    print('⚠️ Chưa hoàn thành hoặc bị lỗi! Không tìm thấy file zip đầu ra.')
print('\n✅ Cell 7 Done!')


## 🧪 Cell 8 — (Tùy chọn) Thử nghiệm cấu hình khác

*Hành động: Dùng để rà quét (Grid Search) các bộ tham số khác nếu muốn thử vận may nghiệm ra điểm số cao hơn nữa.*

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts_Test'

# ── Chỉnh số này ──
HIGH_CONF = 0.68
SAFE      = 0.58
MAX_ART   = 2
# ──────────────────

print(f'🔧 HIGH_CONF={HIGH_CONF}, SAFE={SAFE}, MAX_ART={MAX_ART}')
# Thêm check=True để quẳng lỗi đỏ ra màn hình Colab nếu tiến trình tune bị sập
subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', str(HIGH_CONF), '--safe', str(SAFE),
     '--min-art', '0', '--max-art', str(MAX_ART),
     '--device', 'cuda', '--batch-size', '64', '--reset'],
    capture_output=False,
    check=True
)
src = 'artifacts/output/submission_reranked_test.zip'
tag = f'{HIGH_CONF}_{MAX_ART}_{SAFE}'.replace('.', '')
if os.path.exists(src):
    shutil.copy2(src, f'{DRIVE_DIR}/submission_{tag}.zip')
    print(f'✅ submission_{tag}.zip saved to Drive')
